In [1]:
# ==========================================================
# Notebook 02: Indexing Documents with FAISS
# ==========================================================

import numpy as np
import pandas as pd

from pathlib import Path

from sentence_transformers import SentenceTransformer

import faiss
import pickle

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
corpus_path = "data/search_corpus.csv"

corpus_df = pd.read_csv(corpus_path)

corpus_df.head()

,doc_id,title,category,text,char_length,word_count,source,language,version
0,1,Artificial Intelligence for Beginners,AI,Artificial Intelligence is the simulation of h...,76,10,internal_knowledge_base,en,v1
1,2,Machine Learning Basics,ML,Machine Learning is a subset of Artificial Int...,78,12,internal_knowledge_base,en,v1
2,3,Deep Learning Introduction,DL,Deep Learning uses neural networks with multip...,63,9,internal_knowledge_base,en,v1
3,4,Natural Language Processing,NLP,Natural Language Processing helps computers un...,70,8,internal_knowledge_base,en,v1
4,5,Python Programming,Programming,Python is one of the most popular programming ...,63,11,internal_knowledge_base,en,v1


In [3]:
print("Number of Documents:", len(corpus_df))

print("\nColumns:")

print(corpus_df.columns.tolist())

Number of Documents: 5

Columns:
['doc_id', 'title', 'category', 'text', 'char_length', 'word_count', 'source', 'language', 'version']


In [4]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(model_name)

print("Model Loaded Successfully!")

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model Loaded Successfully!


In [6]:
documents = corpus_df["text"].tolist()

document_embeddings = model.encode(documents, show_progress_bar=True)

print(document_embeddings)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[[-0.02546675 -0.00809782  0.01351516 ...  0.11974406  0.05051943
  -0.04811091]
 [-0.03658456  0.00034257  0.05554105 ...  0.08845802  0.07154419
  -0.04699568]
 [-0.08864576 -0.0601251   0.03765125 ...  0.04305214  0.00531898
   0.043705  ]
 [ 0.02748734  0.01165658  0.06819661 ...  0.10725053  0.03855789
  -0.04503505]
 [-0.05185492 -0.01883095  0.01201108 ...  0.14364077  0.11947443
   0.02467155]]


In [7]:
document_embeddings = np.array(document_embeddings).astype("float32")

print(document_embeddings.dtype)

float32


In [8]:
embedding_dimension = document_embeddings.shape[1]

index = faiss.IndexFlatL2(embedding_dimension)

print(index)

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000001C4B7B61680> >


In [9]:
index.add(document_embeddings)

print("Total Indexed Documents:", index.ntotal)

Total Indexed Documents: 5


In [10]:
metadata = {}

for idx, row in corpus_df.iterrows():

    metadata[idx] = {
        "doc_id": row["doc_id"],
        "title": row["title"],
        "category": row["category"],
        "text": row["text"],
    }

metadata[0]

{'doc_id': 1,
 'title': 'Artificial Intelligence for Beginners',
 'category': 'AI',
 'text': 'Artificial Intelligence is the simulation of human intelligence by machines.'}

In [11]:
Path("indexes").mkdir(exist_ok=True)

faiss.write_index(index, "indexes/faiss_index.bin")

print("FAISS index saved!")

FAISS index saved!


In [12]:
with open("indexes/metadata.pkl", "wb") as file:

    pickle.dump(metadata, file)

print("Metadata saved!")

Metadata saved!


In [13]:
loaded_index = faiss.read_index("indexes/faiss_index.bin")

print("Loaded vectors:", loaded_index.ntotal)

Loaded vectors: 5


In [14]:
with open("indexes/metadata.pkl", "rb") as file:

    loaded_metadata = pickle.load(file)

loaded_metadata[0]

{'doc_id': 1,
 'title': 'Artificial Intelligence for Beginners',
 'category': 'AI',
 'text': 'Artificial Intelligence is the simulation of human intelligence by machines.'}

In [15]:
query = "How can I start learning Artificial Intelligence?"

query_embedding = model.encode([query]).astype("float32")

print(query_embedding.shape)

(1, 384)


In [16]:
top_k = 3

distances, indices = loaded_index.search(query_embedding, top_k)

print("Distances:")

print(distances)

print("\nIndices:")

print(indices)

Distances:
[[0.9316398 1.0777115 1.1574419]]

Indices:
[[0 4 1]]


In [17]:
print("Top Search Results:\n")

for rank, idx in enumerate(indices[0], start=1):

    result = loaded_metadata[idx]

    print(f"Rank {rank}")

    print("Title:", result["title"])

    print("Category:", result["category"])

    print("Text:", result["text"])

    print("-" * 50)

Top Search Results:

Rank 1
Title: Artificial Intelligence for Beginners
Category: AI
Text: Artificial Intelligence is the simulation of human intelligence by machines.
--------------------------------------------------
Rank 2
Title: Python Programming
Category: Programming
Text: Python is one of the most popular programming languages for AI.
--------------------------------------------------
Rank 3
Title: Machine Learning Basics
Category: ML
Text: Machine Learning is a subset of Artificial Intelligence that learns from data.
--------------------------------------------------


In [18]:
def semantic_search(query, model, index, metadata, top_k=3):

    query_embedding = model.encode([query]).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for distance, idx in zip(distances[0], indices[0]):

        doc = metadata[idx]

        results.append(
            {
                "title": doc["title"],
                "category": doc["category"],
                "distance": float(distance),
                "text": doc["text"],
            }
        )

    return pd.DataFrame(results)

In [19]:
semantic_search(
    query="Explain Machine Learning",
    model=model,
    index=loaded_index,
    metadata=loaded_metadata,
    top_k=3,
)

,title,category,distance,text
0,Machine Learning Basics,ML,0.532251,Machine Learning is a subset of Artificial Int...
1,Artificial Intelligence for Beginners,AI,1.012453,Artificial Intelligence is the simulation of h...
2,Deep Learning Introduction,DL,1.117110,Deep Learning uses neural networks with multip...


In [20]:
while True:

    user_query = input("\nEnter query (or 'quit'): ")

    if user_query.lower() == "quit":
        break

    output = semantic_search(
        query=user_query,
        model=model,
        index=loaded_index,
        metadata=loaded_metadata,
        top_k=3,
    )

    print("\nResults:\n")
    print(output)


Results:

                         title     category  distance  \
0  Natural Language Processing          NLP  1.099350   
1           Python Programming  Programming  1.564234   
2      Machine Learning Basics           ML  1.648741   

                                                text  
0  Natural Language Processing helps computers un...  
1  Python is one of the most popular programming ...  
2  Machine Learning is a subset of Artificial Int...  
